In [ ]:
from pathlib import Path
import json
import sys
import time
import uuid

import imageio.v2 as imageio
import numpy as np
import pybullet as p

REPO_ROOT = Path('/home/mzoellner/Projects/research/guided-diffusion')
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'robomimic'))
sys.path.insert(0, str(REPO_ROOT / 'calvin' / 'calvin_env'))

import robomimic.envs
import robomimic.utils.file_utils as FileUtils
from calvin_experiments.calvin_rollout_utils import articulated_binaries_from_start_state, check_state_difference, classify_behavior

CONFIG_PATH = REPO_ROOT / 'calvin_experiments' / 'env_playground_config.json'
SCENE_INDEX = {
    'sliding_door': 0,
    'drawer': 1,
    'button': 2,
    'switch': 3,
    'lightbulb': 4,
    'green_light': 5,
}
BLOCK_POSE_SLICES = {
    'block_red': (6, 9, 9, 12),
    'block_blue': (12, 15, 15, 18),
    'block_pink': (18, 21, 21, 24),
}


def get_calvin_unwrapped_env(env):
    """Return the underlying CALVIN gym env through robomimic wrappers."""
    current = env
    while hasattr(current, 'env'):
        child = current.env
        if hasattr(child, 'unwrapped'):
            return child.unwrapped
        current = child
    raise AttributeError('Could not find underlying CALVIN env')


def refresh_obs_after_visual_change(env):
    """Refresh robomimic observations after notebook-only PyBullet edits."""
    calvin_env = get_calvin_unwrapped_env(env)
    robomimic_env = env.unwrapped if hasattr(env, 'unwrapped') else env
    robomimic_env._current_obs = calvin_env.get_obs()
    base_obs = robomimic_env.get_observation()
    if hasattr(env, '_get_initial_obs_history') and hasattr(env, '_get_stacked_obs_from_history'):
        env.timestep = 0
        env.update_obs(base_obs, reset=True)
        env.obs_history = env._get_initial_obs_history(base_obs)
        return env._get_stacked_obs_from_history()
    return base_obs


def render_visual_camera(env, video_cfg):
    """Notebook-only camera. Does not affect policy observations or env metadata."""
    if video_cfg.get('perspective') == 'third_person':
        return env.render(
            mode='rgb_array',
            height=int(video_cfg['height']),
            width=int(video_cfg['width']),
        )

    calvin_env = get_calvin_unwrapped_env(env)
    width = int(video_cfg['width'])
    height = int(video_cfg['height'])
    view_matrix = p.computeViewMatrix(
        cameraEyePosition=video_cfg['look_from'],
        cameraTargetPosition=video_cfg['look_at'],
        cameraUpVector=video_cfg['up_vector'],
    )
    projection_matrix = p.computeProjectionMatrixFOV(
        fov=float(video_cfg['fov']),
        aspect=width / height,
        nearVal=float(video_cfg['nearval']),
        farVal=float(video_cfg['farval']),
    )
    image = p.getCameraImage(
        width=width,
        height=height,
        viewMatrix=view_matrix,
        projectionMatrix=projection_matrix,
        physicsClientId=calvin_env.cid,
    )
    return np.reshape(image[2], (height, width, 4))[:, :, :3].astype(np.uint8)


def apply_block_scene_overrides(scene, blocks_cfg):
    """Set block poses through scene_obs before reset_to, preserving env structure."""
    applied = []
    for name, pose_cfg in blocks_cfg.get('poses', {}).items():
        if name not in BLOCK_POSE_SLICES:
            print(f'Unknown block name in blocks.poses: {name}')
            continue
        pos_start, pos_end, rot_start, rot_end = BLOCK_POSE_SLICES[name]
        if isinstance(pose_cfg, dict):
            position = pose_cfg.get('position')
            rotation = pose_cfg.get('rotation', [0.0, 0.0, 0.0])
        else:
            position = pose_cfg
            rotation = [0.0, 0.0, 0.0]
        scene[pos_start:pos_end] = np.asarray(position, dtype=np.float32)
        scene[rot_start:rot_end] = np.asarray(rotation, dtype=np.float32)
        applied.append(name)
    return applied


def apply_visual_block_filter(env, blocks_cfg):
    """Notebook-only block hiding. Keeps CALVIN scene metadata untouched."""
    hidden = set(blocks_cfg.get('hide', []))
    if not hidden:
        return []

    calvin_env = get_calvin_unwrapped_env(env)
    hidden_position = np.asarray(blocks_cfg.get('hidden_position', [0.0, 0.0, -2.0]), dtype=np.float32)
    hidden_orientation = p.getQuaternionFromEuler([0.0, 0.0, 0.0])
    hidden_names = []

    for idx, obj in enumerate(calvin_env.scene.movable_objects):
        if obj.name not in hidden:
            continue
        position = hidden_position.copy()
        position[2] -= 0.3 * idx
        p.resetBasePositionAndOrientation(
            obj.uid,
            position.tolist(),
            hidden_orientation,
            physicsClientId=calvin_env.cid,
        )
        p.resetBaseVelocity(
            obj.uid,
            linearVelocity=[0.0, 0.0, 0.0],
            angularVelocity=[0.0, 0.0, 0.0],
            physicsClientId=calvin_env.cid,
        )
        p.setCollisionFilterGroupMask(
            obj.uid,
            -1,
            collisionFilterGroup=0,
            collisionFilterMask=0,
            physicsClientId=calvin_env.cid,
        )
        hidden_names.append(obj.name)

    missing = sorted(hidden - set(hidden_names))
    if missing:
        print(f'Unknown block names in blocks.hide: {missing}')
    return hidden_names


In [ ]:
from matplotlib.patches import Circle, Rectangle


def _pose_from_joint_object(obj):
    link_state = obj.p.getLinkState(obj.uid, obj.joint_index, physicsClientId=obj.cid)
    position = np.asarray(link_state[4], dtype=np.float32)
    orientation = np.asarray(link_state[5], dtype=np.float32)
    return position, orientation


def _pose_from_link_object(obj, link_index):
    link_state = obj.p.getLinkState(obj.uid, link_index, physicsClientId=obj.cid)
    position = np.asarray(link_state[4], dtype=np.float32)
    orientation = np.asarray(link_state[5], dtype=np.float32)
    return position, orientation


def _pose_from_body_object(obj):
    position, orientation = obj.p.getBasePositionAndOrientation(obj.uid, physicsClientId=obj.cid)
    return np.asarray(position, dtype=np.float32), np.asarray(orientation, dtype=np.float32)


def _aabb_from_object(obj, link_index=None):
    if link_index is None:
        lower, upper = obj.p.getAABB(obj.uid, physicsClientId=obj.cid)
    else:
        lower, upper = obj.p.getAABB(obj.uid, link_index, physicsClientId=obj.cid)
    return np.asarray(lower, dtype=np.float32), np.asarray(upper, dtype=np.float32)


def capture_scene_snapshot(env):
    calvin_env = get_calvin_unwrapped_env(env)
    scene = calvin_env.scene
    snapshot = {
        'fixed_objects': [],
        'controls': [],
        'blocks': [],
    }

    for obj in scene.fixed_objects:
        position, orientation = _pose_from_body_object(obj)
        lower, upper = _aabb_from_object(obj)
        snapshot['fixed_objects'].append({
            'name': obj.name,
            'position': position.tolist(),
            'orientation': orientation.tolist(),
            'aabb': [lower.tolist(), upper.tolist()],
        })

    for obj in scene.buttons:
        position, orientation = _pose_from_joint_object(obj)
        lower, upper = _aabb_from_object(obj, obj.joint_index)
        snapshot['controls'].append({
            'kind': 'button',
            'name': obj.name,
            'position': position.tolist(),
            'orientation': orientation.tolist(),
            'aabb': [lower.tolist(), upper.tolist()],
            'state': float(obj.get_state()),
        })

    for obj in scene.doors:
        position, orientation = _pose_from_joint_object(obj)
        lower, upper = _aabb_from_object(obj, obj.joint_index)
        kind = 'drawer' if 'drawer' in obj.name else 'slider' if 'slide' in obj.name else 'door'
        snapshot['controls'].append({
            'kind': kind,
            'name': obj.name,
            'position': position.tolist(),
            'orientation': orientation.tolist(),
            'aabb': [lower.tolist(), upper.tolist()],
            'state': float(obj.get_state()),
        })

    for obj in scene.switches:
        position, orientation = _pose_from_joint_object(obj)
        lower, upper = _aabb_from_object(obj, obj.joint_index)
        snapshot['controls'].append({
            'kind': 'switch',
            'name': obj.name,
            'position': position.tolist(),
            'orientation': orientation.tolist(),
            'aabb': [lower.tolist(), upper.tolist()],
            'state': float(obj.get_state()),
        })

    for obj in scene.lights:
        position, orientation = _pose_from_link_object(obj, obj.link_id)
        lower, upper = _aabb_from_object(obj, obj.link_id)
        snapshot['controls'].append({
            'kind': 'light',
            'name': obj.name,
            'position': position.tolist(),
            'orientation': orientation.tolist(),
            'aabb': [lower.tolist(), upper.tolist()],
            'state': float(obj.get_state()),
        })

    for obj in scene.movable_objects:
        position, orientation = _pose_from_body_object(obj)
        lower, upper = _aabb_from_object(obj)
        snapshot['blocks'].append({
            'name': obj.name,
            'position': position.tolist(),
            'orientation': orientation.tolist(),
            'aabb': [lower.tolist(), upper.tolist()],
        })

    return snapshot


def save_scene_snapshot(snapshot, path):
    with open(path, 'w') as f:
        json.dump(snapshot, f, indent=2)


def _rect_from_aabb(ax, lower, upper, **kwargs):
    width = float(upper[0] - lower[0])
    height = float(upper[1] - lower[1])
    patch = Rectangle((float(lower[0]), float(lower[1])), width, height, **kwargs)
    ax.add_patch(patch)
    return patch


def _circle_from_aabb(ax, lower, upper, **kwargs):
    center = ((float(lower[0]) + float(upper[0])) / 2, (float(lower[1]) + float(upper[1])) / 2)
    radius = 0.5 * min(float(upper[0] - lower[0]), float(upper[1] - lower[1]))
    patch = Circle(center, radius=radius, **kwargs)
    ax.add_patch(patch)
    return patch


def _scene_limits_from_snapshot(snapshot, eef_xy, padding=0.05):
    boxes = []
    for group in ('controls',):
        for item in snapshot.get(group, []):
            if 'aabb' in item:
                lower = np.asarray(item['aabb'][0], dtype=np.float32)
                upper = np.asarray(item['aabb'][1], dtype=np.float32)
                boxes.append((lower, upper))
    if not boxes:
        return (-0.4, 0.4), (-0.4, 0.25)

    lower_xy = np.min([box[0][:2] for box in boxes], axis=0)
    upper_xy = np.max([box[1][:2] for box in boxes], axis=0)
    lower_xy = np.minimum(lower_xy, np.min(eef_xy[:, :2], axis=0))
    upper_xy = np.maximum(upper_xy, np.max(eef_xy[:, :2], axis=0))
    return (float(lower_xy[0] - padding), float(upper_xy[0] + padding)), (float(lower_xy[1] - padding), float(upper_xy[1] + padding))


def draw_scene_snapshot(ax, snapshot, eef_xy=None):
    kind_styles = {
        'button': {'facecolor': '#111111', 'edgecolor': '#000000', 'alpha': 0.95},
        'slider': {'facecolor': '#4c566a', 'edgecolor': '#1f2937', 'alpha': 0.92},
        'drawer': {'facecolor': '#7a5c4d', 'edgecolor': '#2f241f', 'alpha': 0.9},
        'switch': {'facecolor': '#6b7280', 'edgecolor': 'black', 'alpha': 0.9},
        'light': {'facecolor': '#9ca3af', 'edgecolor': 'black', 'alpha': 0.75},
    }

    for control in snapshot.get('controls', []):
        lower = np.asarray(control['aabb'][0], dtype=np.float32)
        upper = np.asarray(control['aabb'][1], dtype=np.float32)
        kind = control['kind']
        style = kind_styles.get(kind, {'facecolor': '#adb5bd', 'edgecolor': 'black', 'alpha': 0.8})
        if kind == 'button':
            _circle_from_aabb(
                ax,
                lower,
                upper,
                facecolor=style['facecolor'],
                edgecolor=style['edgecolor'],
                linewidth=1.0,
                alpha=style['alpha'],
            )
        else:
            _rect_from_aabb(
                ax,
                lower,
                upper,
                facecolor=style['facecolor'],
                edgecolor=style['edgecolor'],
                linewidth=1.0,
                alpha=style['alpha'],
            )
        label_x = float((lower[0] + upper[0]) / 2)
        label_y = float(upper[1] + 0.01)
        ax.text(label_x, label_y, control['name'].replace('base__', ''), ha='center', va='bottom', fontsize=7, color='black')

    if eef_xy is not None and len(eef_xy) > 0:
        ax.plot(eef_xy[:, 0], eef_xy[:, 1], color='#ff3b30', linewidth=2.5, label='EEF XY path')
        ax.scatter(eef_xy[0, 0], eef_xy[0, 1], c='#4b5563', s=55, edgecolors='white', linewidths=1.0, label='start')
        ax.scatter(eef_xy[-1, 0], eef_xy[-1, 1], c='#111827', s=55, edgecolors='white', linewidths=1.0, label='end')

    ax.set_aspect('equal')
    ax.grid(color='white', alpha=0.2, linewidth=0.8)


In [ ]:
import robomimic.utils.torch_utils as TorchUtils

with open(CONFIG_PATH, 'r') as f:
    exp_cfg = json.load(f)

rollout_cfg = exp_cfg['policy_rollout']
checkpoint_path = rollout_cfg.get('checkpoint_path')

if not checkpoint_path:
    print('Set policy_rollout.checkpoint_path in env_playground_config.json before running policy rollouts.')
    policy_rollout_summaries = []
else:
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.is_absolute():
        checkpoint_path = REPO_ROOT / checkpoint_path
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Policy checkpoint not found: {checkpoint_path}')

    output_root = REPO_ROOT / exp_cfg['output_root']
    run_id = time.strftime('%Y%m%d_%H%M%S') + '_policy_' + uuid.uuid4().hex[:8]
    policy_rollout_out_dir = output_root / run_id
    policy_rollout_out_dir.mkdir(parents=True, exist_ok=True)

    video_cfg = exp_cfg.get('visualization_camera', exp_cfg.get('video', {}))
    fps = int(exp_cfg.get('fps', 30))
    horizon = int(rollout_cfg.get('horizon', 100))
    num_rollouts = int(rollout_cfg.get('num_rollouts', 10))
    stop_on_behavior = bool(rollout_cfg.get('stop_on_behavior', True))
    for_display_stop = bool(rollout_cfg.get('for_display_stop', False))

    device = TorchUtils.get_torch_device(try_to_use_cuda=True)
    policy, ckpt_dict = FileUtils.policy_from_checkpoint(
        ckpt_path=str(checkpoint_path),
        device=device,
        verbose=True,
    )

    policy_rollout_summaries = []

    for start_cfg in rollout_cfg['start_configs']:
        start_name = start_cfg['name']
        start_out_dir = policy_rollout_out_dir / start_name
        start_out_dir.mkdir(parents=True, exist_ok=True)

        # Fresh env per start config prevents hidden-block collision masks from
        # leaking into the visible-block condition, while keeping all rollouts
        # within that condition exactly reset to the same robot + scene state.
        env, _ = FileUtils.env_from_checkpoint(
            ckpt_dict=ckpt_dict,
            render=False,
            render_offscreen=True,
            verbose=True,
        )

        try:
            obs = env.reset()
            default_state = env.get_state()
            fixed_robot = np.asarray(default_state['robot'], dtype=np.float32).copy()
            fixed_scene = np.asarray(default_state['scene'], dtype=np.float32).copy()

            for name, value in start_cfg.get('env_setup', {}).items():
                fixed_scene[SCENE_INDEX[name]] = float(value)
            apply_block_scene_overrides(fixed_scene, start_cfg.get('blocks', {}))

            scene_snapshot_path = start_out_dir / 'scene_snapshot.json'
            obs = env.reset_to({'scene': fixed_scene.copy(), 'robot': fixed_robot.copy()})
            hidden_blocks = apply_visual_block_filter(env, start_cfg.get('blocks', {}))
            if hidden_blocks:
                obs = refresh_obs_after_visual_change(env)
            save_scene_snapshot(capture_scene_snapshot(env), scene_snapshot_path)

            for rollout_idx in range(num_rollouts):
                policy.start_episode()
                obs = env.reset_to({'scene': fixed_scene.copy(), 'robot': fixed_robot.copy()})
                hidden_blocks = apply_visual_block_filter(env, start_cfg.get('blocks', {}))
                if hidden_blocks:
                    obs = refresh_obs_after_visual_change(env)

                start_state = env.get_state()
                start_scene = np.asarray(start_state['scene'], dtype=np.float32).copy()
                binaries = articulated_binaries_from_start_state(start_scene)

                frames = [render_visual_camera(env, video_cfg)]
                actions = []
                rewards = []
                dones = []
                scene_states = [start_scene.copy()]
                robot_states = [np.asarray(start_state['robot'], dtype=np.float32).copy()]
                eef_xy = [robot_states[-1][:2].copy()]

                detected_behavior = 'none'
                detected_step = -1

                for step in range(horizon):
                    action = policy(ob=obs)
                    obs, reward, done, info = env.step(action)
                    state = env.get_state()
                    scene = np.asarray(state['scene'], dtype=np.float32).copy()
                    robot = np.asarray(state['robot'], dtype=np.float32).copy()

                    actions.append(np.asarray(action, dtype=np.float32).copy())
                    rewards.append(float(reward))
                    dones.append(bool(done))
                    scene_states.append(scene)
                    robot_states.append(robot)
                    eef_xy.append(robot[:2].copy())
                    frames.append(render_visual_camera(env, video_cfg))

                    if detected_step < 0 and check_state_difference(start_scene, scene, robot[:3], binaries, for_display=for_display_stop):
                        detected_behavior = classify_behavior(start_scene, scene)
                        detected_step = step + 1
                        if stop_on_behavior:
                            break
                    if done:
                        break

                rollout_dir = start_out_dir / f'rollout_{rollout_idx:03d}'
                rollout_dir.mkdir(parents=True, exist_ok=True)
                video_path = rollout_dir / f'{start_name}_rollout_{rollout_idx:03d}_{video_cfg["perspective"]}.mp4'
                trace_path = rollout_dir / 'rollout_trace.npz'

                imageio.mimsave(video_path, frames, fps=fps)
                np.savez_compressed(
                    trace_path,
                    actions=np.asarray(actions, dtype=np.float32),
                    rewards=np.asarray(rewards, dtype=np.float32),
                    dones=np.asarray(dones, dtype=bool),
                    scene_states=np.asarray(scene_states, dtype=np.float32),
                    robot_states=np.asarray(robot_states, dtype=np.float32),
                    eef_xy=np.asarray(eef_xy, dtype=np.float32),
                    detected_behavior=np.asarray(detected_behavior),
                    detected_behavior_step=np.asarray(detected_step, dtype=np.int32),
                    start_config=np.asarray(start_name),
                )

                policy_rollout_summaries.append({
                    'start_config': start_name,
                    'rollout': rollout_idx,
                    'behavior': detected_behavior,
                    'step': detected_step,
                    'video': video_path,
                    'trace': trace_path,
                    'scene_snapshot': scene_snapshot_path,
                })
        finally:
            try:
                get_calvin_unwrapped_env(env).close()
            except Exception:
                pass

    summary_path = policy_rollout_out_dir / 'policy_rollout_summary.json'
    with open(summary_path, 'w') as f:
        json.dump([
            {key: str(value) if isinstance(value, Path) else value for key, value in row.items()}
            for row in policy_rollout_summaries
        ], f, indent=2)

    print(f'Policy rollout output: {policy_rollout_out_dir}')
    print(f'Summary JSON: {summary_path}')
    print('Detected behaviors:')
    for row in policy_rollout_summaries:
        print(f"{row['start_config']} rollout {row['rollout']:03d} -> {row['behavior']} @ step {row['step']} | video: {row['video']}")

    if policy_rollout_summaries:
        try:
            from IPython.display import Video, display
            display(Video(filename=str(policy_rollout_summaries[0]['video']), embed=True))
        except Exception as exc:
            print(f'Inline video display skipped: {exc}')


In [ ]:
from IPython.display import Video, display

if 'policy_rollout_summaries' in globals() and policy_rollout_summaries:
    first_video = policy_rollout_summaries[0]['video']
    print(f'First policy rollout video: {first_video}')
    display(Video(filename=str(first_video), embed=True))
else:
    print('Run the policy rollout cell below first; it will save one video per rollout.')

print('Available visualization perspectives in this notebook:')
print('- freiburg_style: custom PyBullet camera from env_playground_config.json, used only for saved rollout videos')
print('- third_person: CALVIN static camera used by the robomimic image obs / env.render path')
print('- eye_in_hand: wrist camera available in observations, not rendered by this video helper')


In [ ]:
import matplotlib.pyplot as plt

if 'policy_rollout_summaries' not in globals() or not policy_rollout_summaries:
    print('Run the policy rollout cell below first; it writes trace .npz files for plotting.')
else:
    item = policy_rollout_summaries[0]
    with open(item['scene_snapshot'], 'r') as f:
        scene_snapshot = json.load(f)

    trace = np.load(item['trace'])
    eef_xy = trace['eef_xy']

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.set_facecolor('#fbf7ef')
    draw_scene_snapshot(ax, scene_snapshot, eef_xy=eef_xy)
    xlim, ylim = _scene_limits_from_snapshot(scene_snapshot, eef_xy)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_title(f"Top-down rollout | {item['start_config']} #{item['rollout']:02d} | behavior: {item['behavior']} @ step {item['step']}")
    ax.set_xlabel('world x [m]')
    ax.set_ylabel('world y [m]')
    ax.legend(loc='upper right')

    plot_path = Path(item['trace']).with_name('rollout_scene_overlay.png')
    fig.savefig(plot_path, dpi=180, bbox_inches='tight')
    print(f'Saved scene overlay plot: {plot_path}')
    plt.show()
